# Lab 2: Modelos e Deployment (15 min)

## Objetivos
- Compreender o catálogo de modelos do Foundry
- Usar o SDK `azure-ai-inference` para chamadas de chat
- Explorar parâmetros: temperature, max_tokens, system prompt
- Comparar comportamentos do modelo

## Conceitos

### Catálogo de Modelos
O Foundry disponibiliza 1600+ modelos de diferentes fornecedores:
- **OpenAI**: GPT-4o, GPT-4o-mini, o1, o3
- **Meta**: Llama 3.1, 3.2, 3.3
- **Mistral**: Mistral Large, Small
- **Microsoft**: Phi-4, MAI

### Tipos de Deployment
| Tipo | Descrição | Quando usar |
|------|-----------|-------------|
| **Standard** | Pay-as-you-go, partilhado | Desenvolvimento, baixo volume |
| **Provisioned** | Throughput reservado | Produção, latência previsível |
| **Serverless** | Modelos MaaS (pay-per-token) | Modelos de terceiros |

### SDK Unificado
No Foundry v2, usamos `azure-ai-inference` para todas as chamadas a modelos:
```python
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential

client = ChatCompletionsClient(
    endpoint=FOUNDRY_ENDPOINT,
    credential=AzureKeyCredential(FOUNDRY_KEY),
)
```

In [ ]:
# Setup - carregar configuração
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
key = os.getenv("AZURE_AI_FOUNDRY_KEY")
model = os.getenv("MODEL_DEPLOYMENT", "gpt-4o")

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key),
)

print(f"✅ Cliente criado para: {endpoint}")
print(f"   Modelo: {model}")

## 🖥️ Exercício 2.1: Primeira Chamada ao Modelo

Vamos fazer uma chamada simples com mensagem de sistema (system prompt).

In [ ]:
# Exercício 2.1: Chamada com system prompt
response = client.complete(
    model=model,
    messages=[
        SystemMessage(content="És um assistente especialista em Azure. Responde sempre em português de Portugal."),
        UserMessage(content="O que é o Azure AI Foundry? Explica em 3 frases."),
    ],
    max_tokens=200,
    temperature=0.7,
)

print("=== Resposta ===")
print(response.choices[0].message.content)
print(f"\n📊 Tokens: prompt={response.usage.prompt_tokens}, resposta={response.usage.completion_tokens}, total={response.usage.total_tokens}")

## 🖥️ Exercício 2.2: Streaming

Em aplicações reais, usamos streaming para mostrar o texto à medida que é gerado.

In [ ]:
# Exercício 2.2: Streaming de resposta
print("=== Streaming ===")

response = client.complete(
    model=model,
    messages=[
        SystemMessage(content="És um poeta português."),
        UserMessage(content="Escreve um haiku sobre cloud computing."),
    ],
    max_tokens=100,
    stream=True,
)

for chunk in response:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

print("\n\n✅ Streaming concluído!")

## 🖥️ Exercício 2.3: Conversação Multi-turn

Os modelos mantêm contexto quando envias o histórico da conversa.

In [ ]:
# Exercício 2.3: Conversação com histórico
from azure.ai.inference.models import AssistantMessage

historico = [
    SystemMessage(content="És um assistente técnico de Azure. Responde de forma concisa."),
    UserMessage(content="O que é um Resource Group?"),
]

# Primeira pergunta
r1 = client.complete(model=model, messages=historico, max_tokens=150)
resposta1 = r1.choices[0].message.content
print("👤 O que é um Resource Group?")
print(f"🤖 {resposta1}\n")

# Adicionar ao histórico e fazer follow-up
historico.append(AssistantMessage(content=resposta1))
historico.append(UserMessage(content="E como se relaciona com uma subscription?"))

r2 = client.complete(model=model, messages=historico, max_tokens=150)
print("👤 E como se relaciona com uma subscription?")
print(f"🤖 {r2.choices[0].message.content}")

## 🖥️ Exercício 2.4: Comparar Temperature

A **temperature** controla a aleatoriedade das respostas:
- `0.0` → Determinístico, factual
- `1.0` → Criativo, variado

In [ ]:
# Exercício 2.4: Comparar temperatures
prompt = "Sugere um nome para uma startup de IA em saúde."

for temp in [0.0, 0.5, 1.0]:
    response = client.complete(
        model=model,
        messages=[UserMessage(content=prompt)],
        max_tokens=50,
        temperature=temp,
    )
    print(f"🌡️ Temperature {temp}: {response.choices[0].message.content.strip()}")
    print()

## 🧪 Desafio Extra

Experimenta com diferentes system prompts para ver como mudam as respostas:
- `"És um pirata que só fala em rimas"`
- `"Responde sempre com exatamente 3 bullet points"`
- `"És um engenheiro senior que revê código"`

In [ ]:
# Desafio: experimenta aqui!
response = client.complete(
    model=model,
    messages=[
        SystemMessage(content="Responde sempre com exatamente 3 bullet points"),
        UserMessage(content="Quais as vantagens de usar Azure AI Foundry?"),
    ],
    max_tokens=200,
)

print(response.choices[0].message.content)

## ✅ Resumo

Neste lab aprendeste:
- A usar o `ChatCompletionsClient` do `azure-ai-inference`
- Chamadas simples, streaming e multi-turn
- O impacto de temperature e system prompts
- Que o SDK é o mesmo independentemente do modelo

**Próximo:** [Lab 3 - Agentes](lab03-agentes.ipynb)